# L13c Example: NanoGPT on Tiny Shakespeare
This example trains a small character-level decoder-only language model on the Tiny Shakespeare corpus and samples from it. The goal is to make every component of the L13c lecture concrete: a real causal-attention layer, a real next-token-prediction training loop, and real autoregressive sampling with greedy, temperature, and top-$k$ decoding.

> __Problem statement.__ We have a single text file, the complete works of Shakespeare concatenated into about $1.1$ million characters. We want to train a small language model (around $112{,}000$ parameters, two transformer blocks, four heads) on this file by next-token prediction, and then use it to generate new Shakespeare-like text starting from a prompt. The trained model will not produce real Shakespearean prose, because it is small and the corpus is short, but it will reproduce the surface structure: speech tags, character names, line breaks, and roughly word-shaped sequences of characters.

The rest of this notebook works toward that goal in three tasks: preparing the corpus, building and training the model, and sampling from it.

> __Learning Objectives:__
>
> By the end of this example, you should be able to:
>
> * __Build and train a decoder-only language model from scratch:__ Assemble a NanoGPT model from the `CausalAttention`, `DecoderBlock`, and `NanoGPT` building blocks in `src/`, and train it on Tiny Shakespeare with next-token prediction cross-entropy loss.
> * __Generate text with three sampling strategies:__ Use greedy decoding, temperature sampling, and top-$k$ sampling on the trained model and explain by inspection how each strategy trades off determinism against diversity.
> * __Read a causal attention pattern:__ Plot the attention weights of one of the trained heads on a real input sequence and identify the lower-triangular structure imposed by the causal mask.

Let's get started!
___

## Setup
We load the L13c Julia environment, which brings in `Flux`, `NNlib`, `JLD2`, our custom `Shakespeare.jl`, `CausalAttention.jl`, `DecoderBlock.jl`, `NanoGPT.jl`, and `Sample.jl` files in the `src/` directory.

In [1]:
include("Include.jl");
Random.seed!(42);

  Activating project at `~/Desktop/julia_work/CHEME-5820-instances/Spring-2026/CHEME-5820-Lectures-Spring-2026/lectures/week-13/L13c`

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while tryin

## Task 1: Prepare the Tiny Shakespeare corpus
The first task is to turn the raw text file into integer tensors the model can consume. We (i) download the corpus from the char-rnn GitHub repo (cached locally after the first run), (ii) build a character-level vocabulary that maps each unique character to an integer id, (iii) encode the full corpus as an integer vector, and (iv) hold out the last 10% as a validation set so we can monitor for overfitting.

The cell below runs all four steps in one `let ... end` block and returns three named bindings:

* `vocab` — a `Vocabulary` object with `char_to_id`, `id_to_char`, and `vocab_size` fields.
* `TRAIN_DATA` — the integer-encoded training portion of the corpus.
* `VAL_DATA` — the integer-encoded validation portion of the corpus.

In [2]:
vocab, TRAIN_DATA, VAL_DATA = let

    # (i) load corpus from cache or download
    path = download_shakespeare(_PATH_TO_DATA);
    text = read(path, String);
    @info "corpus loaded" path n_chars=length(text)
    println();
    println("First 200 characters of the corpus:");
    println("---");
    println(text[1:200]);
    println("---");

    # (ii) build character-level vocabulary
    vocab = build_vocab(text);
    @info "vocabulary built" vocab_size=vocab.vocab_size
    println("Vocab characters: \"", join(sort(collect(keys(vocab.char_to_id))), ""), "\"")

    # encode/decode roundtrip sanity check
    test_str = "First Citizen:";
    ids = encode(test_str, vocab);
    println("encoded \"", test_str, "\" -> ", ids);
    println("decoded back: \"", decode(ids, vocab), "\"");

    # (iii) encode full corpus and (iv) 90/10 split
    data = encode(text, vocab);
    n_train = round(Int, 0.9 * length(data));
    TRAIN_DATA = data[1:n_train];
    VAL_DATA   = data[n_train+1:end];
    @info "train/val split" n_train=length(TRAIN_DATA) n_val=length(VAL_DATA)

    # return -
    (vocab, TRAIN_DATA, VAL_DATA)
end

┌ Info: corpus loaded
│   path = /Users/jeffreyvarner/Desktop/julia_work/CHEME-5820-instances/Spring-2026/CHEME-5820-Lectures-Spring-2026/lectures/week-13/L13c/data/input.txt
│   n_chars = 1115394
└ @ Main /Users/jeffreyvarner/Desktop/julia_work/CHEME-5820-instances/Spring-2026/CHEME-5820-Lectures-Spring-2026/lectures/week-13/L13c/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X44sZmlsZQ==.jl:6



First 200 characters of the corpus:
---
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you
---


┌ Info: vocabulary built
│   vocab_size = 65
└ @ Main /Users/jeffreyvarner/Desktop/julia_work/CHEME-5820-instances/Spring-2026/CHEME-5820-Lectures-Spring-2026/lectures/week-13/L13c/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X44sZmlsZQ==.jl:15


Vocab characters: "
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz"
encoded "First Citizen:" -> [19, 48, 57, 58, 59, 2, 16, 48, 59, 48, 65, 44, 53, 11]
decoded back: "First Citizen:"


┌ Info: train/val split
│   n_train = 1003855
│   n_val = 111539
└ @ Main /Users/jeffreyvarner/Desktop/julia_work/CHEME-5820-instances/Spring-2026/CHEME-5820-Lectures-Spring-2026/lectures/week-13/L13c/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X44sZmlsZQ==.jl:29


(CharVocabulary(Dict('n' => 53, 'f' => 45, 'w' => 62, 'E' => 18, 'Z' => 39, 'o' => 54, '\'' => 6, 'B' => 15, 'C' => 16, ':' => 11…), Dict(5 => '&', 56 => 'q', 16 => 'C', 20 => 'G', 35 => 'V', 55 => 'p', 60 => 'u', 30 => 'Q', 19 => 'F', 32 => 'S'…), 65), [19, 48, 57, 58, 59, 2, 16, 48, 59, 48  …  54, 52, 44, 58, 2, 47, 44, 57, 44, 13], [1, 1, 20, 31, 18, 26, 22, 28, 11, 1  …  59, 2, 62, 40, 50, 48, 53, 46, 9, 1])

## Task 2: Build and train the NanoGPT model
The second task is to assemble a decoder-only language model and train it on the corpus prepared in Task 1. The model has two transformer blocks, four attention heads, model dimension $64$, and a context length of $64$ characters. The expected parameter count is $2 V d + n_{\max} d + L\,(12 d^{2} + 9 d) + 2 d = 112{,}000$ for $V = 65$, $d = 64$, $L = 2$, $n_{\max} = 64$.

We train with Adam at learning rate $3\times 10^{-3}$ for $5$ epochs of $200$ iterations each, sampling random `(input, target)` chunks from the training corpus on each iteration. The first run trains from scratch and saves a checkpoint to `data/nanogpt_d64_h4_l2.jld2`; subsequent runs detect the checkpoint and load it instead of retraining.

The cell below runs the build-and-train step in one `let ... end` block and returns three named bindings:

* `model` — a trained `NanoGPT` instance.
* `train_history` — vector of per-epoch mean training losses.
* `val_history` — vector of per-epoch validation losses.

In [ ]:
model, train_history, val_history = let

    # hyperparameters
    VOCAB     = vocab.vocab_size;
    D_MODEL   = 64;
    N_HEADS   = 4;
    N_LAYERS  = 2;
    CTX_LEN   = 64;
    D_FF      = 4 * D_MODEL;
    BATCHSIZE = 32;

    # build the model
    Random.seed!(42);
    model = NanoGPT(VOCAB, D_MODEL, N_HEADS, N_LAYERS, CTX_LEN; d_ff = D_FF);
    @info "NanoGPT model built" n_params=n_parameters(model) vocab=VOCAB d_model=D_MODEL n_heads=N_HEADS n_layers=N_LAYERS ctx_len=CTX_LEN

    # training setup
    CHECKPOINT_PATH   = joinpath(_PATH_TO_DATA, "nanogpt_d64_h4_l2.jld2");
    N_EPOCHS          = 5;
    N_ITERS_PER_EPOCH = 200;
    LR                = 3.0f-3;

    train_history = Float32[];
    val_history   = Float32[];

    # load checkpoint if it exists, otherwise train from scratch
    if isfile(CHECKPOINT_PATH)
        @info "Loading pre-trained checkpoint" path=CHECKPOINT_PATH
        saved = JLD2.load(CHECKPOINT_PATH);
        Flux.loadmodel!(model, saved["model_state"]);
        train_history = saved["train_history"];
        val_history   = saved["val_history"];
        @info "Loaded model from checkpoint"
    else
        @info "No checkpoint found, training from scratch..."
        Random.seed!(42);
        opt = Flux.setup(Adam(LR), model);
        for epoch in 1:N_EPOCHS
            epoch_losses = Float32[];
            for it in 1:N_ITERS_PER_EPOCH
                Xb, Yb = sample_batch(TRAIN_DATA, BATCHSIZE, CTX_LEN);
                l, gs = Flux.withgradient(model) do m
                    nanogpt_loss(m, Xb, Yb)
                end
                push!(epoch_losses, l);
                Flux.update!(opt, model, gs[1]);
            end
            train_l = mean(epoch_losses);
            push!(train_history, train_l);
            Xv, Yv = sample_batch(VAL_DATA, BATCHSIZE, CTX_LEN);
            val_l = nanogpt_loss(model, Xv, Yv);
            push!(val_history, val_l);
            @info "epoch" epoch train_loss=round(train_l, digits=4) val_loss=round(val_l, digits=4)
        end
        JLD2.jldsave(CHECKPOINT_PATH;
                      model_state = Flux.state(model),
                      train_history = train_history,
                      val_history = val_history);
        @info "Saved checkpoint" path=CHECKPOINT_PATH
    end

    # return -
    (model, train_history, val_history)
end

### Training loss curve
Before moving on, we sanity-check the training run by plotting the per-epoch training and validation losses. Both should decrease steadily and stay close together, which means the model is learning the corpus without overfitting.

In [ ]:
let
    plot(1:length(train_history), train_history;
         xlabel = "epoch", ylabel = "cross-entropy loss",
         label = "train",
         title = "NanoGPT training on Tiny Shakespeare",
         linewidth = 2, marker = :circle, markersize = 5,
         legend = :topright);
    plot!(1:length(val_history), val_history;
          label = "validation",
          linewidth = 2, marker = :diamond, markersize = 5)
end

## Task 3: Sample from the model and visualize causal attention
The third task is to use the trained model. We generate text with three decoding strategies (greedy, temperature, and top-$k$), and then pull the attention weights out of one head of the first decoder block and plot them as a heatmap to confirm the causal-mask pattern from the lecture.

All three sampling strategies share the same prompt, `"ROMEO:"`, and the same generation length, $250$ characters, so any differences come from the decoding rule, not the prompt.

### Greedy Decoding ($\tau = 0$)
At every step, pick the token with the highest logit. This is deterministic, but it tends to fall into repetition loops because the most-likely continuation of "the the the" is often "the" again.

In [ ]:
println(sample_text(model, vocab, "ROMEO:", 250; temperature = 0))

### Temperature Sampling ($\tau = 0.8$)
Scale the logits by $1/\tau$ before softmax. With $\tau = 0.8$ the distribution stays sharp but is no longer deterministic, so the model can escape repetition loops.

In [ ]:
println(sample_text(model, vocab, "ROMEO:", 250; temperature = 0.8, rng = MersenneTwister(7)))

### Top-$k$ Sampling ($\tau = 0.8$, $k = 10$)
Same as temperature sampling but restricted to the $k$ highest-logit tokens at every step. This avoids occasional very-low-probability "junk" tokens that pure temperature sampling can produce.

In [ ]:
println(sample_text(model, vocab, "ROMEO:", 250; temperature = 0.8, top_k = 10, rng = MersenneTwister(7)))

### Causal attention heatmap
Each attention head in the model produces a $T\times T$ matrix of attention weights at each forward pass. Because of the causal mask, every row $i$ has zero weight at all columns $j > i$, so the matrix is strictly lower-triangular. We pick a random input sequence from the validation set, run the embeddings and the first decoder block's pre-norm on it, and call `causal_attention_weights` to recover the $(T, T, H, 1)$ attention tensor from the first decoder block.

The cell below runs this extraction in one `let ... end` block and returns two named bindings:

* `attn_w` — the attention-weight tensor of shape $(T, T, H, 1) = (64, 64, 4, 1)$.
* `input_chars` — the decoded character string of the input sequence, used for the axis labels.

In [ ]:
attn_w, input_chars = let

    # pick a sample input from the validation set
    Random.seed!(2026);
    Xb, _ = sample_batch(VAL_DATA, 1, model.ctx_len);
    input_chars = decode(Xb[:, 1], vocab);
    println("Input sequence:");
    println("---");
    println(input_chars);
    println("---");

    # embed and pre-norm the input, then pull attention weights from block 1, head 1
    d_model = size(model.tok_emb, 1);
    T       = model.ctx_len;
    flat_ids = vec(Xb);
    tok_e    = model.tok_emb[:, flat_ids];
    H0       = reshape(tok_e, d_model, T, 1) .+ reshape(model.pos_emb[:, 1:T], d_model, T, 1);
    H_norm   = model.blocks[1].ln1(H0);
    attn_w   = causal_attention_weights(model.blocks[1].attn, H_norm);
    @info "attention weights" shape=size(attn_w)

    # return -
    (attn_w, input_chars)
end

In [ ]:
let
    # plot the attention weights for head 1
    A = attn_w[:, :, 1, 1];
    T = size(A, 1);
    labels = [c == '\n' ? "\\n" : string(c) for c in input_chars];
    tick_labels = ["$(i): $(labels[i])" for i in 1:T];
    heatmap(1:T, 1:T, A;
            xlabel = "key (attended-to position)",
            ylabel = "query (attending position)",
            title  = "Causal attention weights, layer 1, head 1",
            color  = :viridis,
            size   = (800, 700),
            yflip  = true,
            xticks = (1:T, tick_labels),
            yticks = (1:T, tick_labels),
            xrotation = 75,
            xtickfont = font(5),
            ytickfont = font(5))
end

### Things to Think About
The plot above shows you exactly what the causal mask buys: every row of the heatmap has nonzero values only in the lower-left triangle, which is the structural constraint we wanted from the lecture. A few questions to think through:

* __What does each row of the heatmap mean?__ Row $i$ is the attention distribution that the model uses when computing the output at position $i$. Look at one row of the lower-triangular region. Where does the model put most of its attention? On the immediately preceding token, on a few specific earlier tokens, or spread evenly?
* __Why are the entries in the upper-right triangle exactly zero?__ The causal mask sets the score at every $(i, j)$ with $j > i$ to $-10^{9}$ before softmax, and $\exp(-10^{9})$ rounds to zero. This is the empirical confirmation that the mask is doing its job: information from the future cannot leak into the past, no matter what the input contains.
* __What did greedy decoding produce, and why is it bad?__ Look at the greedy sample above and compare it to the temperature samples. Greedy decoding always picks the same continuation given the same context, so once it enters a repeating pattern it can never escape. Temperature sampling and top-$k$ sampling break this loop by injecting randomness, at the cost of reproducibility.
* __What has the model NOT learned?__ Read one of the temperature samples carefully. The model has learned line breaks, speech tags ("ROMEO:" / "MENENIUS:"), and roughly word-shaped sequences of characters, but it has not learned actual word spellings, grammar, or semantics. What would you change about the experiment to push the model toward more coherent output? (Larger model, more training, larger context window, word-level rather than character-level tokenization, more data, or a combination of these.)
* __How does parameter count scale with model dimension?__ The dominant term in the decoder-stack parameter count is $L\cdot 12 d^{2}$. If you double $d$ from 64 to 128, by what factor does the per-block parameter count increase? Verify your prediction by counting parameters of a $d=128$ model with the same $L$ and $H$.

## Summary
We trained a small decoder-only language model on Tiny Shakespeare and sampled from it with three different decoding strategies. Every component of the L13c lecture, the causal mask, the decoder block, the embedding tables, and the next-token cross-entropy loss, became something concrete you could read off the model and inspect.

> __Key Takeaways:__
>
> * **Small decoder-only LM is enough to see the lecture's claims in practice:** With $\sim$112k parameters and 1000 gradient steps on Tiny Shakespeare, the model already produces output with the right surface structure (line breaks, character names, speech tags), even though the actual word content is gibberish.
> * **Greedy decoding fails for the reason the lecture predicted:** Picking the highest-logit token at every step drops the model into a repetition loop almost immediately, and temperature and top-$k$ sampling fix this by injecting controlled randomness.
> * **Causal attention heatmap is the most direct visualization of the causal mask:** A trained head's attention weights are exactly lower-triangular with all upper-triangular entries equal to zero, which is the architectural constraint that enables parallel training of an autoregressive model.
___